# Lab 15/16 — Speculative decoding, measured

Build the speculative-decoding rule yourself — first on **dummy distributions** (where correctness is
provable), then as a **real two-model stitch** (gpt2 verified against distilgpt2, a same-tokenizer pair).
Read the [lab brief](Lab15_16.md) first. Marked on **relationships between your own numbers**; the
sampling parts are seeded from your roll number, so your numbers are yours.

**Before you run anything:** set your identity below. Fill every 📝 cell. Run top to bottom, then run the
final export cell and submit the two files it names. CPU is enough (`gpt2` + `distilgpt2`).

In [1]:
ROLL_NUMBER = "202518030"      # <- my roll number
NAME        = "Dhruv Parmar"   # <- my name

assert ROLL_NUMBER and NAME, "Set ROLL_NUMBER and NAME before running the rest."


## Setup — dummy data + the model pair (given)

In [2]:
import json, platform, sys, hashlib
from pathlib import Path
import numpy as np
import torch, torch.nn.functional as F
import transformers
from transformers import AutoTokenizer, AutoModelForCausalLM
from transformers.utils import logging as hf_logging
hf_logging.set_verbosity_error(); hf_logging.disable_progress_bar()

SEED = int(hashlib.sha256(ROLL_NUMBER.encode()).hexdigest(), 16) % (2**31)   # your seed
rng = np.random.default_rng(SEED)

# --- dummy distributions (Parts 1-3): 5-symbol toy, one row per drafted position (+ bonus row) ---
VOCAB = ["cat", "sat", "on", "the", "mat"]; V = len(VOCAB); k = 3
p = np.array([[0.50,0.20,0.10,0.10,0.10],
              [0.10,0.60,0.10,0.10,0.10],
              [0.10,0.10,0.20,0.50,0.10],
              [0.20,0.20,0.20,0.20,0.20]])   # (k+1, V) target; row k = bonus
q = np.array([[0.70,0.10,0.10,0.05,0.05],
              [0.10,0.50,0.20,0.10,0.10],
              [0.10,0.10,0.60,0.10,0.10]])   # (k, V) draft (over-proposes cat, on)
drafts = [0, 1, 2]
assert np.allclose(p.sum(1),1) and np.allclose(q.sum(1),1)

# --- the real model pair (Part 4): same tokenizer, both small, CPU-fast ---
TARGET, DRAFT = "gpt2", "distilgpt2"
tokenizer = AutoTokenizer.from_pretrained(TARGET)
target = AutoModelForCausalLM.from_pretrained(TARGET).eval()
draft  = AutoModelForCausalLM.from_pretrained(DRAFT).eval()
assert target.config.vocab_size == draft.config.vocab_size, "the stitch needs a shared vocabulary"

RESULTS = {}
print("dummy vocab", V, "| models", TARGET, "<-", DRAFT, "| shared vocab", target.config.vocab_size, "| seed", SEED)

dummy vocab 5 | models gpt2 <- distilgpt2 | shared vocab 50257 | seed 881596549


---
## Part 1 — The acceptance branch (dummy)

Accept a drafted token `x` at position `i` with probability `min(1, p[i,x] / q[i,x])`.

📝 **Predict.** When is the accept probability exactly **1**? Position 0 drafted `"cat"` with `q=0.70`,
`p=0.50` — what is its accept probability, and why does over-proposing get **throttled**?

**Prediction.** The accept probability is exactly **1** whenever `p[i,x] >= q[i,x]` — the drafter
proposed that token no more often than the target wants it, so there is nothing to correct and the
ratio is clamped by the `min(1, .)`.

Position 0 drafted `"cat"` with `q = 0.70` against `p = 0.50`, so the ratio is `0.50/0.70 = 5/7`
and the accept probability is **≈ 0.714**. Over-proposing gets throttled because the drafter is
supplying that token `q/p = 1.4x` too often, and accepting only a `p/q` fraction of those proposals
brings the committed rate back down to exactly `p`: `q x (p/q) = p`. The correction is multiplicative,
so it removes precisely the excess and nothing more.

I expect the empirical rates to land within about 0.005 of `min(1, p/q)` at 20,000 draws per position,
and position 1 (`"sat"`, `q = 0.50` against `p = 0.60`) to sit at exactly 1.0, since the drafter
UNDER-proposes there and no throttling is needed.


In [3]:
def is_accepted(p_i, q_i, x, u):
    """Ratio test for ONE drafted token. u is a uniform(0,1) draw passed in for reproducibility.
    Accept with probability min(1, p_i[x] / q_i[x]); return True/False."""
    ratio = p_i[x] / q_i[x] if q_i[x] > 0 else float("inf")
    return bool(u < min(1.0, ratio))


# harness (given): empirical accept rate per position should track min(1, p/q)
emp = [float(np.mean([is_accepted(p[i], q[i], drafts[i], u) for u in rng.random(20000)])) for i in range(k)]
theo = [float(min(1.0, p[i, drafts[i]]/q[i, drafts[i]])) for i in range(k)]
RESULTS["accept"] = dict(positions=list(range(k)), empirical=[round(x,3) for x in emp],
                         theoretical=[round(x,3) for x in theo])
for i in range(k): print(f"pos {i} ({VOCAB[drafts[i]]}): empirical {emp[i]:.3f}  vs  min(1,p/q) {theo[i]:.3f}")

pos 0 (cat): empirical 0.717  vs  min(1,p/q) 0.714
pos 1 (sat): empirical 1.000  vs  min(1,p/q) 1.000
pos 2 (on): empirical 0.332  vs  min(1,p/q) 0.333


📝 **Explain.** What is embarrassingly parallel here and what is not? Tie it to Lecture 15/16's
"verification is a prefill".

**Explanation.** Embarrassingly parallel: **the `k` ratio tests**. Each position `i` compares `u[i]`
against `min(1, p[i, x_i] / q[i, x_i])`, and none of those comparisons needs any other one's result — so
the whole test is a gather, a divide and a compare over length-`k` arrays. More importantly, the `p` rows
they read were *all produced by a single target forward pass*, so the expensive work behind them was
already done in parallel too.

Not parallel: **the cut, and the commit**. Position `i` only *counts* if every earlier position was
accepted, so turning the accept vector into `n_accepted` is inherently a prefix operation (here
`np.argmin(accept)` — the first `False` — guarded by `accept.all()` for the no-reject case), and the token
emitted at the cut depends on where the cut fell. Everything scored past the cut is discarded.

That is exactly the shape of "verification is a prefill". The costly half — one weight read scoring `k+1`
positions — is the parallel, compute-bound half, and it is charged once per block no matter how many
positions ride along. The sequential half is pure book-keeping over `k` booleans, and costs nothing.
Speculation works because the serial dependency was moved off the expensive resource and onto a free one.

The agreement with the scalar reference is **1.0000 over 3000 checks** — and it
should be exactly 1, not merely close, because both paths consume the *same* `u` vector and apply the same
comparison. This check is algebraic, not statistical: anything below 1.0 would mean the vectorised cut and
the loop disagree about which position rejected first, which is a bug rather than noise.


---
## Part 2 — Rejection sampling: the residual (dummy)

A reject resamples from the **residual** `norm(max(0, p − q))`, **not** from `p`. That correction is what
makes the emitted stream exactly `p` — and a version that resamples from `p` has the *same acceptance rate*
but a silently wrong output.

📝 **Predict.** Over many single-step emissions, what distribution should the first emitted token follow?
What will the **broken** version (resample from `p`) get wrong that the acceptance rate cannot reveal?

**Prediction.** The first emitted token should follow **`p[0]` exactly** — `[0.50, 0.20, 0.10, 0.10, 0.10]` —
because the accept branch delivers `min(p, q)` of the mass and the residual branch delivers the rest.
I expect a total-variation distance of a few thousandths at 60,000 draws, which is sampling noise.

The **broken** version resamples from `p` on a reject instead of from `norm(max(0, p - q))`. It gets the
**mixture weights** wrong: tokens the drafter over-proposes (`cat` at 0.70 against 0.50, `on` at 0.60
against 0.20) already arrive through the accept branch at their full `p` rate, and then get a second
helping from the rejection draw — so they end up over-represented, and the under-proposed tokens are
starved. I expect `cat` to come out noticeably above 0.50 and the TV distance to be an order of
magnitude larger than the correct version.

Crucially the **acceptance rate is identical** in both: the bug lives entirely on the rejection branch,
which fires exactly as often either way. Nothing in a latency or acceptance dashboard can see it.


In [4]:
def residual(p_i, q_i):
    """Distribution to sample from after a rejection: proportional to max(0, p_i - q_i), renormalised."""
    r = np.maximum(0.0, np.asarray(p_i, dtype=float) - np.asarray(q_i, dtype=float))
    total = r.sum()
    # total == 0 only if q dominates p everywhere, which forces sum(q) > sum(p); impossible for
    # two normalised distributions unless p == q, in which case nothing is ever rejected anyway.
    return np.asarray(p_i, dtype=float).copy() if total <= 0 else r / total

def sample_on_reject(p_i, q_i):
    """Draw a replacement token id from residual(p_i, q_i) using rng."""
    return int(rng.choice(V, p=residual(p_i, q_i)))


# reference single-position emitter (given): draft x ~ q0, accept/reject, emit ~ p0
def emit_once(sampler):
    x = int(rng.choice(V, p=q[0]))
    if is_accepted(p[0], q[0], x, rng.random()):
        return x
    return sampler(p[0], q[0])              # rejected -> replacement from the residual

def broken_sampler(p_i, q_i):               # the silent bug: resample from p, not the residual
    return int(rng.choice(V, p=p_i))

def dist_of(sampler, n=60000):
    c = np.zeros(V)
    for _ in range(n): c[emit_once(sampler)] += 1
    return c / c.sum()

emp_p   = dist_of(sample_on_reject)
brk_p   = dist_of(broken_sampler)
tv = lambda a, b: 0.5*float(np.abs(a-b).sum())
RESULTS["emit"]   = dict(target_p0=[round(x,3) for x in p[0]], empirical=[round(x,3) for x in emp_p],
                         tv_distance=round(tv(emp_p, p[0]), 4))
RESULTS["broken"] = dict(tv_distance=round(tv(brk_p, p[0]), 4))
print("target p0        :", np.round(p[0],3))
print("correct emitted  :", np.round(emp_p,3), " TV =", RESULTS["emit"]["tv_distance"])
print("broken  emitted  :", np.round(brk_p,3), " TV =", RESULTS["broken"]["tv_distance"], " <- should be far larger")

target p0        : [0.5 0.2 0.1 0.1 0.1]
correct emitted  : [0.5   0.2   0.099 0.1   0.101]  TV = 0.0014
broken  emitted  : [0.598 0.139 0.122 0.07  0.07 ]  TV = 0.1205  <- should be far larger


📝 **Explain (the paragraph that carries this part).** Why the residual and not `p`? An implementation
that resamples from `p` has an *identical acceptance rate* — name what it gets wrong and why only a
distribution test (not a latency or acceptance dashboard) catches it.

**Explanation.** A committed token arrives through one of two branches, and together they have to
reconstruct `p` exactly. The accept branch delivers `min(p, q)` of the mass: the drafter proposes `x` at
rate `q(x)` and it survives with probability `min(1, p(x)/q(x))`, so the product is `min(p(x), q(x))`.
That leaves **`p - min(p, q) = max(0, p - q)`** unaccounted for — precisely the mass on tokens the drafter
did not supply often enough. The rejection branch has to deliver exactly that shortfall and nothing else,
which is why the replacement is drawn from `norm(max(0, p - q))`. Resampling from `p` instead hands out a
second full copy of `p` on the rejection branch, so tokens the drafter already over-supplied through
acceptance get topped up again, and the under-supplied ones stay starved.

The measurement shows exactly that skew. Target `p[0]` is `[0.5, 0.2, 0.1, 0.1, 0.1]`; the
correct emitter produced `[0.5, 0.2, 0.099, 0.1, 0.101]` with **TV = 0.0014**, while
the broken control landed at **TV = 0.1205** — about 86x
worse. The direction is diagnostic, not random: `cat` (over-proposed at 0.70 against 0.50) came out at
0.598 instead of 0.50, and `the` and `mat` (under-proposed) were pushed down to 0.07 from 0.10.

**And the acceptance rate is identical in both**, because the bug lives *entirely inside* the rejection
branch — it changes what is emitted after a reject, never how often a reject happens. So every operational
signal a serving team normally watches is unchanged: same throughput, same acceptance dashboard, same
latency, fluent output. Only a **distribution test** — many draws from a fixed `(p, q)` pair, compared
against `p` by TV or chi-square — can see it, because the defect *is* a distributional one. This is the
sharpest example in the course of a bug that is invisible to every measurement except the one that asks
the question the guarantee actually makes.


---
## Part 3 — Vectorised verification (dummy)

Verification is **one pass over k+1 positions**, so the `k` ratio tests should be array ops, not a python
loop; only the *cut* at the first reject stays sequential.

📝 **Predict.** How do you turn a boolean accept-vector into the **first-reject index** in one step? Which
part must stay sequential, and how does that mirror "verification is a prefill"?

**Prediction.** With `accept` as a boolean vector of length `k`, the first-reject index is
`np.argmin(accept)` — `argmin` returns the first `False` — guarded by `accept.all()`, which means
"no reject at all" and yields `n_accepted = k` plus the bonus draw. So the whole ratio test is three
array ops (gather `p[i,x]` and `q[i,x]`, divide, compare against `u`) and the search for the cut is one more.

What must stay sequential is **the commitment**, not the test: the emitted token at the cut depends on
*where* the cut is, and everything after the cut is discarded, so those positions cannot be committed in
parallel even though they were already scored.

That split is exactly "verification is a prefill". The k+1 positions are scored in **one pass against one
weight read**, which is the parallel, compute-bound half; the sequential half is only the cheap
book-keeping of deciding how much of that work to keep. The expensive resource is spent in parallel and
the serial dependency is pushed onto something that costs nothing.

I expect agreement with the scalar reference to be **1.0000** — this is not a statistical claim but an
algebraic one, since both consume the same `u` and apply the same comparison.


In [5]:
def verify_vectorised(p, q, drafts, u):
    """Verify all k drafts at once. p:(k+1,V) q:(k,V) drafts:(k,) u:(k,) uniforms.
    Return (n_accepted, next_token): n_accepted in 0..k; next_token is a residual draw at the
    cut, or a bonus draw from p[k] if all k accepted. Vectorise the TEST; the cut is sequential."""
    d = np.asarray(drafts)
    kk = len(d)
    rows = np.arange(kk)
    p_x, q_x = p[rows, d], q[rows, d]                       # the k drafted probabilities, both models
    ratio = np.divide(p_x, q_x, out=np.full(kk, np.inf), where=q_x > 0)
    accept = np.asarray(u) < np.minimum(1.0, ratio)         # ALL k ratio tests, one array op
    # the CUT is the only sequential part: first False, or k if none.
    n_accepted = kk if accept.all() else int(np.argmin(accept))
    if n_accepted == kk:
        return n_accepted, int(rng.choice(V, p=p[kk]))      # bonus token, free from the same pass
    return n_accepted, sample_on_reject(p[n_accepted], q[n_accepted])


# harness (given): must agree with a scalar reference built on your Part 1-2 functions
def verify_scalar(p, q, drafts, u):
    for i in range(k):
        if u[i] < min(1.0, p[i, drafts[i]]/q[i, drafts[i]]): continue
        return i, sample_on_reject(p[i], q[i])
    return k, int(rng.choice(V, p=p[k]))

agree = 0; N = 3000
for _ in range(N):
    u = rng.random(k)
    na_v, _ = verify_vectorised(p, q, drafts, u)
    na_s, _ = verify_scalar(p, q, drafts, u)   # compare the CUT (n_accepted); the token draw is random
    agree += (na_v == na_s)
RESULTS["vector"] = dict(agreement=round(agree/N, 4), n_checks=N)
print(f"vectorised vs scalar cut agreement: {agree/N:.4f}  ({N} checks)")

vectorised vs scalar cut agreement: 1.0000  (3000 checks)


📝 **Explain.** What is embarrassingly parallel here and what is not? Tie it to Lecture 15/16's
"verification is a prefill".

**Explanation.** Embarrassingly parallel: **the `k` ratio tests**. Each position `i` compares `u[i]`
against `min(1, p[i, x_i] / q[i, x_i])`, and none of those comparisons needs any other one's result — so
the whole test is a gather, a divide and a compare over length-`k` arrays. More importantly, the `p` rows
they read were *all produced by a single target forward pass*, so the expensive work behind them was
already done in parallel too.

Not parallel: **the cut, and the commit**. Position `i` only *counts* if every earlier position was
accepted, so turning the accept vector into `n_accepted` is inherently a prefix operation (here
`np.argmin(accept)` — the first `False` — guarded by `accept.all()` for the no-reject case), and the token
emitted at the cut depends on where the cut fell. Everything scored past the cut is discarded.

That is exactly the shape of "verification is a prefill". The costly half — one weight read scoring `k+1`
positions — is the parallel, compute-bound half, and it is charged once per block no matter how many
positions ride along. The sequential half is pure book-keeping over `k` booleans, and costs nothing.
Speculation works because the serial dependency was moved off the expensive resource and onto a free one.

The agreement with the scalar reference is **1.0000 over 3000 checks** — and it
should be exactly 1, not merely close, because both paths consume the *same* `u` vector and apply the same
comparison. This check is algebraic, not statistical: anything below 1.0 would mean the vectorised cut and
the loop disagree about which position rejected first, which is a bug rather than noise.


---
## Part 4 — The real stitch: gpt2 verified against distilgpt2

Now the same rule on real models, in the **greedy** case (`T=0`), where correctness is *exactly* checkable:
speculative-greedy output must equal plain-greedy output, token for token — in **fewer target passes**.

Greedy collapses the accept test to an equality: accept the draft iff it equals the target's argmax.

📝 **Predict.** Will the two token streams be identical? Will speculative decoding use fewer **target
forward passes** than tokens generated? Why is the block yield bounded in `[1, k+1]`?

**Prediction.** **Yes, identical.** At `T = 0` the target's distribution collapses to a one-hot vector,
so the ratio test degenerates to an equality check: a draft is kept only if it *is* the target's argmax,
and at the first mismatch the target's own argmax is emitted. Every committed token is therefore the one
plain greedy would have produced, and the context going into the next block is identical too — so the
two streams cannot diverge.

**Yes, fewer target passes than tokens.** Each block costs exactly **one** target pass and commits
between 1 and k+1 tokens, so passes < tokens whenever a single draft is ever accepted. With `k = 4` and
a prompt as predictable as "The capital of France is", I expect distilgpt2 to agree with gpt2 fairly
often — maybe 10-14 passes for 24 tokens.

**The bound is `[1, k+1]`** because a reject still emits the target's own token (the floor of 1, which is
what plain decoding would have produced anyway), and a fully accepted block gets the **bonus** token from
position `k` — the pass already computed that distribution, so it is free. A block can never yield zero.

**Wall-clock is a separate question.** Each block also runs `k` drafter passes. distilgpt2 is 6 layers
against gpt2's 12, so a draft pass is only about half a target pass — 4 drafts cost roughly 2 target
passes. If the acceptance length is below ~3 the arithmetic will not work out, and I would not be
surprised to measure speculative decoding as **slower** here despite using fewer target passes.


In [6]:
PROMPT, N_TOKENS, K = "The capital of France is", 24, 4
ids0 = tokenizer(PROMPT, return_tensors="pt").input_ids

def plain_greedy(ids, n):                    # reference (given): n target passes, n tokens
    ids = ids.clone()
    for _ in range(n):
        with torch.no_grad(): nt = target(ids).logits[0,-1].argmax()
        ids = torch.cat([ids, nt.view(1,1)], 1)
    return ids[0, -n:].tolist()

def speculative_greedy(ids, n, k):
    """Greedy speculative decode. Return (tokens, target_passes, accept_lengths):
       - draft k tokens autoregressively from `draft` (argmax each step)
       - ONE `target` forward pass over the k drafts -> logits at each position
       - accept while target's argmax == the draft; at the first mismatch emit target's argmax and stop;
         if all k accepted, emit the bonus (target's argmax at position k)
       - count ONE target pass per block; record the accepted-run length per block
    """
    ids = ids.clone()
    tokens, acc_lens, target_passes, draft_passes = [], [], 0, 0

    while len(tokens) < n:
        # --- draft k tokens autoregressively from the small model (k cheap passes) -------------
        d_ids, drafts = ids.clone(), []
        for _ in range(k):
            with torch.no_grad():
                nt = draft(d_ids).logits[0, -1].argmax()
            d_ids = torch.cat([d_ids, nt.view(1, 1)], 1)
            drafts.append(int(nt))
            draft_passes += 1

        # --- ONE target pass over context + all k drafts (this is the prefill) -----------------
        with torch.no_grad():
            logits = target(d_ids).logits[0]
        target_passes += 1

        # logits[L-1+i] is the target's distribution for the token at draft position i,
        # so one pass gives k+1 predictions: k to check, plus the bonus at position k.
        L = ids.shape[1]
        tgt = [int(logits[L - 1 + i].argmax()) for i in range(k + 1)]

        # --- accept while the target's argmax agrees; stop at the first mismatch ---------------
        n_acc = 0
        while n_acc < k and tgt[n_acc] == drafts[n_acc]:
            n_acc += 1
        acc_lens.append(n_acc)

        # the token after the accepted run: the target's own argmax (the bonus when n_acc == k),
        # so a block always commits between 1 and k+1 tokens and never zero.
        new = drafts[:n_acc] + [tgt[n_acc]]
        ids = torch.cat([ids, torch.tensor(new).view(1, -1)], 1)
        tokens.extend(new)

    return tokens[:n], target_passes, acc_lens


plain = plain_greedy(ids0, N_TOKENS)
spec, passes, acc_lens = speculative_greedy(ids0, N_TOKENS, K)
RESULTS["stitch"] = dict(target=TARGET, draft=DRAFT, k=K, n_tokens=N_TOKENS,
                         identical_to_greedy=bool(spec[:N_TOKENS] == plain),
                         target_passes=int(passes),
                         accept_lengths=[int(a) for a in acc_lens])
print("identical to plain greedy:", RESULTS["stitch"]["identical_to_greedy"])
print(f"target passes: {passes} for {N_TOKENS} tokens  (plain greedy would use {N_TOKENS})")
print("accepted-run lengths per block:", acc_lens)
print(f"mean accepted run: {sum(acc_lens)/len(acc_lens):.2f}  |  tokens per target pass (acceptance length): {N_TOKENS/passes:.2f}")

identical to plain greedy: True
target passes: 6 for 24 tokens  (plain greedy would use 24)
accepted-run lengths per block: [4, 1, 4, 4, 4, 2]
mean accepted run: 3.17  |  tokens per target pass (acceptance length): 4.00


### Extra: what the template does not measure

The grading fields record *target passes*, which is the right number for the mechanism — but the
write-up asks about **wall clock**, and that also contains `k` drafter passes per block. The next two
cells measure both, plus the per-pass cost of each model, and then sweep `k` so acceptance length and
throughput can be watched coming apart. Nothing here changes the graded fields.


In [7]:
# EXTRA (mine, not in the template): the write-up asks whether this is actually FASTER, so measure it.
# Also count the drafter's own passes -- the term the "fewer target passes" headline leaves out.
import time

def timed(fn, reps=7):
    ts = []
    for _ in range(reps):
        t0 = time.perf_counter(); out = fn(); ts.append(time.perf_counter() - t0)
    return out, min(ts)          # min over reps: least contaminated by OS scheduling

def spec_with_draft_count(ids, n, k):
    """Same algorithm as speculative_greedy, instrumented to also return the drafter pass count."""
    ids = ids.clone(); tokens, acc_lens, tp, dp = [], [], 0, 0
    while len(tokens) < n:
        d_ids, drafts = ids.clone(), []
        for _ in range(k):
            with torch.no_grad(): nt = draft(d_ids).logits[0, -1].argmax()
            d_ids = torch.cat([d_ids, nt.view(1, 1)], 1); drafts.append(int(nt)); dp += 1
        with torch.no_grad(): logits = target(d_ids).logits[0]
        tp += 1
        L = ids.shape[1]; tgt = [int(logits[L - 1 + i].argmax()) for i in range(k + 1)]
        n_acc = 0
        while n_acc < k and tgt[n_acc] == drafts[n_acc]: n_acc += 1
        acc_lens.append(n_acc)
        new = drafts[:n_acc] + [tgt[n_acc]]
        ids = torch.cat([ids, torch.tensor(new).view(1, -1)], 1); tokens.extend(new)
    return tokens[:n], tp, dp, acc_lens

_,  t_plain = timed(lambda: plain_greedy(ids0, N_TOKENS))
_,  t_spec  = timed(lambda: speculative_greedy(ids0, N_TOKENS, K))
_, _, dpass, _ = spec_with_draft_count(ids0, N_TOKENS, K)

# cost of one pass of each model, measured directly, to explain the wall-clock result
_, t_one_target = timed(lambda: target(ids0).logits, reps=15)
_, t_one_draft  = timed(lambda: draft(ids0).logits,  reps=15)

RESULTS["timing"] = dict(
    plain_s=round(t_plain, 3), spec_s=round(t_spec, 3),
    speedup=round(t_plain / t_spec, 3),
    target_passes=int(passes), draft_passes=int(dpass),
    acceptance_length=round(N_TOKENS / passes, 3),
    one_target_pass_ms=round(t_one_target * 1e3, 2),
    one_draft_pass_ms=round(t_one_draft * 1e3, 2),
    draft_cost_ratio=round(t_one_draft / t_one_target, 3),
)
print(f"plain greedy      : {t_plain:.3f} s  ({N_TOKENS} target passes)")
print(f"speculative greedy: {t_spec:.3f} s  ({passes} target passes + {dpass} draft passes)")
print(f"wall-clock speedup: {t_plain/t_spec:.2f}x   |  acceptance length: {N_TOKENS/passes:.2f} tokens/pass")
print(f"one pass: target {t_one_target*1e3:.1f} ms  draft {t_one_draft*1e3:.1f} ms  "
      f"(draft costs {t_one_draft/t_one_target:.2f} of a target pass)")


plain greedy      : 0.461 s  (24 target passes)
speculative greedy: 0.406 s  (6 target passes + 24 draft passes)
wall-clock speedup: 1.14x   |  acceptance length: 4.00 tokens/pass
one pass: target 14.9 ms  draft 9.6 ms  (draft costs 0.65 of a target pass)


In [8]:
# EXTRA (mine): acceptance length is not throughput. Sweep k and watch them come apart.
sweep = []
for k_try in (1, 2, 4, 8):
    toks, tp, dp, al = spec_with_draft_count(ids0, N_TOKENS, k_try)
    _, t = timed(lambda kk=k_try: speculative_greedy(ids0, N_TOKENS, kk), reps=7)
    sweep.append(dict(k=k_try, target_passes=tp, draft_passes=dp,
                      acceptance_length=round(N_TOKENS / tp, 3),
                      mean_accepted_run=round(sum(al) / len(al), 3),
                      wall_s=round(t, 3), speedup_vs_plain=round(t_plain / t, 3),
                      identical=bool(toks == plain)))
RESULTS["k_sweep"] = sweep

print(f"{'k':>3} {'tgt passes':>11} {'draft passes':>13} {'accept len':>11} {'wall s':>8} {'speedup':>8}  identical")
for r in sweep:
    print(f"{r['k']:>3} {r['target_passes']:>11} {r['draft_passes']:>13} "
          f"{r['acceptance_length']:>11.2f} {r['wall_s']:>8.3f} {r['speedup_vs_plain']:>8.2f}  {r['identical']}")


  k  tgt passes  draft passes  accept len   wall s  speedup  identical
  1          13            13        1.85    0.410     1.12  True
  2           9            18        2.67    0.395     1.17  True
  4           6            24        4.00    0.406     1.14  True
  8           3            24        8.00    0.348     1.32  True


📝 **Explain (the write-up).** Acceptance ≠ speedup: on this CPU, is speculative decoding actually
*faster* in wall-clock than plain greedy? Fewer target passes, yet the drafter runs its own passes —
reconcile the two. And: why does a bad drafter cost speed but **never** accuracy?

**Explanation (the write-up).**

**Was it faster? Yes, but far less than the pass count suggests.** Plain greedy took
**0.461 s** for 24 tokens; speculative greedy took **0.406 s** —
a wall-clock speedup of **1.14x**. Yet it used only **6 target passes
instead of 24**, i.e. an acceptance length of **4.00 tokens per
target pass**. A 4x reduction in target passes bought a 1.14x reduction in time. My
prediction that it might come out *slower* was wrong in sign but right in cause: the drafter's own passes
eat almost all of the win.

**Reconciling the two numbers.** The headline counts only target passes; the clock also pays for
**24 drafter passes**. Measured directly on this machine, one target pass costs
**14.8 ms** and one drafter pass **9.6 ms**, so a draft
costs **0.65 of a target pass** — distilgpt2 is half the layers of gpt2 but nothing like half the cost,
because at 124M parameters on a CPU the per-call overhead does not halve with depth. Per block, then:

```
cost  = 1 target pass + k drafts = 1 + k x 0.65   (in target-pass units)
yield = acceptance length L tokens
speedup = L / (1 + k x 0.65)
```

which is the lecture's `acceptance length / (1 + overhead)` with the overhead named. At k=4:
`4.00 / (1 + 4 x 0.65) = 1.11x`
against a measured 1.14x. The sweep confirms the model across the range:

| k | target passes | draft passes | acceptance length | wall s | measured | model `L/(1+kr)` |
|---|---|---|---|---|---|---|
| 1 | 13 | 13 | 1.85 | 0.410 | **1.12x** | 1.12x |
| 2 | 9 | 18 | 2.67 | 0.395 | **1.17x** | 1.16x |
| 4 | 6 | 24 | 4.00 | 0.406 | **1.14x** | 1.11x |
| 8 | 3 | 24 | 8.00 | 0.348 | **1.32x** | 1.30x |

Every row lands within a few percent of the model, so the reasoning is not a story fitted after the fact.

**Acceptance length is not throughput, and this table is the proof.** Acceptance length rises
**monotonically** with `k` (1.85 → 8.00) while the
speedup does **not** (1.12 → 1.17 →
1.14 → 1.32): k=2 beats k=4 despite a shorter
acceptance run, because `L` grows **sub-linearly** in `k` while the drafting overhead `k x r` grows
**linearly**. k=8 wins here only because this prompt is so predictable that the drafter was accepted
every single time (8.00 = k, three blocks, nothing discarded) — that is a
property of "The capital of France is", not a general result, and it would collapse on open-ended prose.

**Two honest caveats about the regime.** (i) Neither implementation uses a **KV cache**: every pass
re-runs the whole prefix, so both arms are more expensive than a real server and the drafter's share grows
with the sequence. (ii) More fundamentally, speculation exists to sell **idle compute during a
memory-bound decode**, and a 124M model on a CPU is not in that regime — the surplus it monetises barely
exists here. On a 27B target at batch 1 the drafter would be a few percent of a target pass rather than
65%, and the same acceptance length would convert into something close to the full 4.0x.

**Why a bad drafter costs speed but never accuracy.** Correctness never depended on the drafter. At `T=0`
the test is an equality against the target's argmax, so a rejected draft is simply replaced by the token
the target wanted — the same token plain greedy would have emitted — and the output came out **identical
to plain greedy, token for token** (True), with per-block accepted runs
[4, 1, 4, 4, 4, 2], every one inside `[0, 4]`. Under sampling the same holds through the residual
rather than the argmax. So a drafter that agrees rarely only means more discarded proposals: the block
still commits between 1 and k+1 tokens — never zero, because a reject still emits the target's own token —
and in the limit of zero acceptance the algorithm degenerates to plain decoding **plus** wasted drafting.
**The downside is bounded at slightly slower; it is never subtly wrong.** That asymmetry is why
speculation can be switched on before it has been measured, and it is exactly what quantisation or
distillation cannot promise.


---
## Submit

Run this last. It writes `submission_lab15_16_<roll>.json`. Submit **two files**: that JSON and this
executed notebook with every 📝 cell filled.

In [9]:
env = dict(platform=platform.platform(), python=sys.version.split()[0],
           torch=torch.__version__, transformers=transformers.__version__,
           numpy=np.__version__, seed=SEED, target=TARGET, draft=DRAFT,
           linux=sys.platform.startswith("linux"))
sub = dict(roll=ROLL_NUMBER, name=NAME, env=env, results=RESULTS)
out = Path(f"submission_lab15_16_{ROLL_NUMBER}.json")
out.write_text(json.dumps(sub, indent=2))
print("wrote", out, "| parts:", list(RESULTS))
assert set(RESULTS) >= {"accept","emit","broken","vector","stitch"}, "run every part before exporting"

wrote submission_lab15_16_202518030.json | parts: ['accept', 'emit', 'broken', 'vector', 'stitch', 'timing', 'k_sweep']
